In [1]:
import os, sys, subprocess, gc, time
import torch

# Auto-detect Base Directory (Supports C:\Users\DIVE_GUEST\LJH\ecommerce_journey)
potential_dirs = [
    r"C:\Users\DIVE_GUEST\LJH\ecommerce_journey\predict",
    r"C:\Users\DIVE_GUEST\LJH\ecommerce_journey",
    os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd(),
    r"c:\Users\user\Desktop\마케팅도메인지식기반 구매예측_논문\장바구니 이탈 연구 선행 논문\새 폴더\predict"
]

base_dir = os.getcwd()
for p_dir in potential_dirs:
    if os.path.exists(os.path.join(p_dir, "xgb.py")):
        base_dir = p_dir
        break

# Auto-detect Output JSON Directory
out_dir = r"C:\Users\DIVE_GUEST\LJH\predict"
if not os.path.exists(r"C:\Users\DIVE_GUEST\LJH") and os.path.exists(r"D:\LJH"):
    out_dir = r"D:\LJH\predict"
elif not os.path.exists(os.path.dirname(out_dir)):
    out_dir = os.path.join(base_dir, "results")

os.makedirs(out_dir, exist_ok=True)

models = [
    "xgb.py",
    #"RF.py",
    #"MLP.py",
    #"CNN.py",
    #"RNN.py",
    #"LSTM.py",
    #"CNNLSTM.py",
    #"RNNLSTM.py",
    #"tabnet.py"
]

print(f"[*] Base Model Directory: {base_dir}")
print(f"[*] Output JSON Directory : {out_dir}")
print(f"[*] Total Models to Run   : {len(models)}")
print("=" * 65)

execution_summary = {}

for idx, script in enumerate(models, 1):
    in_path = os.path.join(base_dir, script)
    if not os.path.exists(in_path):
        print(f"[SKIP] File not found: {in_path}")
        continue
        
    print(f"\n🚀 [{idx}/{len(models)}] Executing: {script}")
    print("-" * 65)
    sys.stdout.flush()
    
    start_t = time.time()
    
    cmd = [sys.executable, "-u", in_path]
    
    try:
        process = subprocess.Popen(
            cmd,
            cwd=base_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            encoding='utf-8',
            errors='replace'
        )
        
        for line in iter(process.stdout.readline, ''):
            print(line, end='')
            sys.stdout.flush()
            
        process.stdout.close()
        return_code = process.wait()
        elapsed = time.time() - start_t
        
        if return_code == 0:
            print(f"\n✅ [{script}] Successfully Finished ({elapsed:.1f}s)")
            execution_summary[script] = f"SUCCESS ({elapsed:.1f}s)"
        else:
            print(f"\n⚠️ [{script}] Exited with return code {return_code} ({elapsed:.1f}s)")
            execution_summary[script] = f"FAILED (code {return_code})"
            
    except Exception as e:
        print(f"\n💥 Error executing {script}: {e}")
        execution_summary[script] = f"ERROR ({e})"
        
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    gc.collect()
    time.sleep(2)

print("\n=======================================================")
print("       ALL PREDICTION MODELS AUTOMATION COMPLETE        ")
print("=======================================================")
for script, status in execution_summary.items():
    print(f"  • {script:<15}: {status}")


[*] Base Model Directory: /mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey
[*] Output JSON Directory : /mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey/results
[*] Total Models to Run   : 1

🚀 [1/1] Executing: xgb.py
-----------------------------------------------------------------
[DEVICE] Detected device for XGBoost: cuda
[LOAD] Loading dataset from: /mnt/d/LJH/data/final_data_100k_64.parquet
[MEMORY OPTIMIZATION] Downcasting feature dtypes to float32/int16...
[DATASET CHECK] Total Rows: 9,999,329 | Positives: 12,340 | Target Imbalance Ratio: 0.123408%
[COST-SENSITIVE] Full Train Set Negative/Positive Ratio = 809.32

  [STEP 1: OPTUNA] Individual Tuning for 'base' (41 Features)
[base OPTUNA DONE] Best params: {
  "scale_pos_weight": 113.17838155876477,
  "n_estimators": 100,
  "learning_rate": 0.282207813054926,
  "max_depth": 8,
  "min_child_weight": 10,
  "subsample": 0.7506896548455161,
  "colsample_bytree": 0.9663374661012957,
  "gamma": 4.791114420052891,
  "reg_alpha": 0.006117530988